# 02 · EDA: network, demand trends and the RASK KPI

In [ ]:
import sys, json, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
from src.metrics import load_clean, headline, DATA_DIR, OUTPUT_DIR, DATA_END
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
NAVY, VIOLET, LILAC, ORANGE, SLATE, GREEN = "#030734", "#682EFF", "#CD83FF", "#F05100", "#62748E", "#009588"
sns.set_theme(style="whitegrid", palette=[VIOLET, LILAC, NAVY, ORANGE, SLATE, GREEN])
FIG = OUTPUT_DIR / "figures"; FIG.mkdir(parents=True, exist_ok=True); DATA_DIR.mkdir(parents=True, exist_ok=True)
Cr = lambda v: f"₹{v/1e7:,.2f} Cr"; Lk = lambda v: f"₹{v/1e5:,.1f} L"
findings = []
def note(q, topic, text):
    findings.append(dict(q=q, topic=topic, finding=text)); print(f"[{q}] {topic}: {text}")
def save(fig, name):
    fig.savefig(FIG / f"{name}.png", dpi=150, bbox_inches="tight"); plt.show()
def kpis(g):
    return pd.Series({"flights": len(g), "revenue": g.Revenue.sum(), "pax": g.PaxCount.sum(), "seats": g.Capacity.sum(),
                      "ask": g.ask.sum(), "rpk": g.rpk.sum(), "lf": g.PaxCount.sum() / g.Capacity.sum(),
                      "fare": g.Revenue.sum() / g.PaxCount.sum(), "rask": g.Revenue.sum() / g.ask.sum(),
                      "yield_rpk": g.Revenue.sum() / g.rpk.sum(), "stage_km": g.ask.sum() / g.Capacity.sum()})

In [ ]:
f = pd.read_parquet(DATA_DIR / "flights.parquet")
NET = kpis(f)

## Q1 · Network and route performance

In [ ]:
R = f.groupby("route").apply(kpis, include_groups=False)
R["share"] = R.revenue / R.revenue.sum(); R = R.sort_values("revenue", ascending=False)
R["quadrant"] = np.select([(R.lf >= NET.lf) & (R.rask >= NET.rask), (R.lf < NET.lf) & (R.rask >= NET.rask),
                           (R.lf >= NET.lf) & (R.rask < NET.rask)], ["Star", "Fill seats", "Raise yield"], "Review route")
Q = R.groupby("quadrant").agg(routes=("share", "size"), revenue_share=("share", "sum"), lf=("lf", "mean"), rask=("rask", "median"))
display(R.head(10)[["flights", "revenue", "share", "lf", "fare", "rask", "stage_km", "quadrant"]]); display(Q)
R.to_csv(OUTPUT_DIR / "route_scorecard.csv")
n80 = int((R.share.cumsum() < .8).sum() + 1)
A = f.groupby("Origin").apply(kpis, include_groups=False).sort_values("revenue", ascending=False); A["share"] = A.revenue / A.revenue.sum()
hub = f[f.Origin.isin(["PNQ", "BOM", "BLR"]) | f.Destination.isin(["PNQ", "BOM", "BLR"])].shape[0] / len(f)
note("Q1", "Concentration", f"Top 10 routes earn {R.share.head(10).sum():.0%} of revenue; {n80} of {len(R)} routes make 80%. "
     f"{(R.flights < 100).sum()} thin routes (<100 flights) add only {R.share[R.flights < 100].sum():.1%}.")
note("Q1", "Hubs", f"Pune is the main hub: {A.share['PNQ']:.0%} of revenue departs PNQ, then Bengaluru {A.share['BLR']:.0%} and Delhi {A.share['DEL']:.0%}. "
     f"{hub:.0%} of flights touch PNQ, BOM or BLR.")
note("Q1", "Route quadrants", " | ".join(f"{q}: {int(r.routes)} routes, {r.revenue_share:.0%} of revenue" for q, r in Q.iterrows()))
rv = R[R.quadrant == "Review route"].head(4)
note("Q1", "Routes to review", "Largest routes below the network on both load factor and RASK: " +
     ", ".join(f"{i} ({r.share:.1%} of revenue, LF {r.lf:.0%}, RASK ₹{r.rask:.3f})" for i, r in rv.iterrows()))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for q, c in zip(["Star", "Fill seats", "Raise yield", "Review route"], [VIOLET, LILAC, NAVY, ORANGE]):
    s = R[R.quadrant == q]; ax.scatter(s.lf, s.rask.clip(upper=.6), s=s.revenue / 2e5, c=c, alpha=.7, label=f"{q} ({len(s)})")
ax.axvline(NET.lf, color=SLATE, ls="--", lw=1); ax.axhline(NET.rask, color=SLATE, ls="--", lw=1)
ax.set_xlabel("Load factor"); ax.set_ylabel("RASK (₹ per seat-km, capped at 0.6)"); ax.legend()
ax.xaxis.set_major_formatter(lambda v, _: f"{v:.0%}"); ax.set_title("Route quadrants vs network average"); save(fig, "e01_route_quadrants")
c = np.corrcoef(np.log(R.stage_km), np.log(R.rask))[0, 1]
note("Q1", "Stage length", f"Longer routes earn less per seat-km (log correlation {c:.2f}). Compare RASK between routes of similar length.")

## Q2 · Demand trends

In [ ]:
Mo = f.groupby("month").apply(kpis, include_groups=False)
Y = f[f.date.dt.month <= 5].groupby(f.date.dt.year).apply(kpis, include_groups=False)
yoy = Y.loc[2025] / Y.loc[2024] - 1
display(Y.T)
fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
ax[0].bar(Mo.index, Mo.pax / 1e3, width=20, color=VIOLET); ax[0].set_title("Passengers per month ('000)")
ax2 = ax[0].twinx(); ax2.plot(Mo.index, Mo.fare, color=ORANGE, marker="o"); ax2.set_ylabel("Avg fare (₹)"); ax2.grid(False)
ax[1].plot(Mo.index, Mo.lf, marker="o", color=NAVY); ax[1].set_title("Load factor per month"); ax[1].yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
save(fig, "e02_monthly_trend")
pre, post = f[f.date < "2024-11-01"], f[f.date >= "2024-11-01"]
kp, kq = kpis(pre), kpis(post)
rp = f.assign(per=np.where(f.date >= "2024-11-01", "post", "pre")).groupby(["route", "per"]).apply(
    lambda g: g.Revenue.sum() / g.PaxCount.sum(), include_groups=False).unstack().dropna()
note("Q2", "Growth Jan-May 2025 vs 2024", f"Flights {yoy.flights:+.0%}, passengers {yoy.pax:+.0%}, revenue {yoy.revenue:+.0%}, "
     f"average fare {yoy.fare:+.0%}, load factor {(Y.loc[2025].lf - Y.loc[2024].lf) * 100:+.1f} pp, RASK {yoy.rask:+.0%}.")
note("Q2", "Fare step Nov 2024", f"Average fare rose from ₹{kp.fare:,.0f} (Jan-Oct 2024) to ₹{kq.fare:,.0f} from Nov 2024 ({kq.fare/kp.fare-1:+.0%}) "
     f"while load factor also rose ({kp.lf:.1%} to {kq.lf:.1%}). {(rp.post / rp.pre > 1.2).mean():.0%} of routes raised fares 20%+. Demand did not fall.")
lo = Mo.loc["2024-07-01":"2024-10-01"]
note("Q2", "Seasonality", f"Weakest stretch Jul-Oct 2024: load factor {lo.pax.sum()/lo.seats.sum():.1%} and RASK ₹{lo.revenue.sum()/lo.ask.sum():.3f} "
     f"vs network {NET.lf:.1%} / ₹{NET.rask:.3f}. Strongest month: {Mo.rask.idxmax():%b %Y} (RASK ₹{Mo.rask.max():.3f}).")
wd = f.groupby(f.date.dt.dayofweek).PaxCount.mean(); chi = stats.kruskal(*[g.PaxCount.values for _, g in f.groupby(f.date.dt.dayofweek)])
note("Q2", "Weekday", f"Passengers per flight range only {wd.min():.1f}-{wd.max():.1f} across the week: the difference is statistically real "
     f"(Kruskal p={chi.pvalue:.3g}, 40k flights) but only {wd.max()/wd.min()-1:.0%} from lowest to highest day. Not worth scheduling around.")

## Q3 · KPI: RASK and its drivers

In [ ]:
Mk = f.groupby("market").apply(kpis, include_groups=False); display(Mk.T)
rask25, rask24 = Y.loc[2025].rask, Y.loc[2024].rask
# decomposition of RASK change: RASK = seat-km LF x yield per RPK (log split)
lf_c = np.log((Y.loc[2025].rpk / Y.loc[2025].ask) / (Y.loc[2024].rpk / Y.loc[2024].ask))
yl_c = np.log(Y.loc[2025].yield_rpk / Y.loc[2024].yield_rpk)
note("Q3", "RASK 2025", f"Jan-May 2025 RASK ₹{rask25:.3f} vs ₹{rask24:.3f} a year earlier ({rask25/rask24-1:+.0%}); target (+5%) ₹{rask24*1.05:.3f}: on target. "
     f"Yield explains {yl_c/(yl_c+lf_c):.0%} of the gain, seat-km load factor {lf_c/(yl_c+lf_c):.0%}.")
note("Q3", "Domestic vs international", f"International is {Mk.loc['International','flights']/NET.flights:.0%} of flights and {Mk.loc['International','revenue']/NET.revenue:.0%} of revenue, "
     f"but load factor {Mk.loc['International','lf']:.0%} vs {Mk.loc['Domestic','lf']:.0%} and RASK ₹{Mk.loc['International','rask']:.3f} vs ₹{Mk.loc['Domestic','rask']:.3f} "
     f"(stage {Mk.loc['International','stage_km']:,.0f} km vs {Mk.loc['Domestic','stage_km']:,.0f} km).")

In [ ]:
F = pd.DataFrame(findings); F.to_csv(OUTPUT_DIR / "eda_findings.csv", index=False)
json.dump({"net": NET.to_dict(), "yoy": yoy.to_dict(), "Y": Y.to_dict(), "quadrants": Q.to_dict(), "pre": kp.to_dict(), "post": kq.to_dict(),
           "market": Mk.to_dict()}, open(OUTPUT_DIR / "eda_numbers.json", "w"), default=float, indent=1)
pd.set_option("display.max_colwidth", None); F